In [0]:
# ============================================================
# Silver — Source 05: AWS SQS Order Events
#
# Transformations:
#   - Cast event_ts ISO string to timestamp
#   - Normalise event_type, order_status, channel
#   - Reject null message_id or order_id → quarantine
#   - customer_id null is valid (some events lack customer context)
#   - amount_pence null is valid (some event types have no amount)
#   - Deduplicate on message_id
#
# Source:  bronze.src_05_sqs.order_events
# Target:  silver.src_05_sqs.order_events
# Quarantine: silver.quarantine.src_05_sqs
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_05_sqs.order_events'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_05_sqs'

VALID_EVENT_TYPES = ['order.placed', 'order.confirmed', 'order.processing',
                     'order.shipped', 'order.delivered', 'order.cancelled', 'order.refunded']
VALID_STATUSES = ['pending', 'confirmed', 'processing', 'shipped', 'delivered', 'cancelled', 'refunded']

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_05_sqs')
print('Silver Source 05 SQS — starting...')


In [0]:
# ── LOAD AND CLEAN ────────────────────────────────────────────
bronze = spark.table(f'{BRONZE_CATALOG}.src_05_sqs.order_events')
total = bronze.count()
print(f'Bronze rows: {total}')

# Step 1: Cast timestamp
df = bronze.withColumn('event_ts', F.to_timestamp(F.col('event_ts')))

# Step 2: Normalise
df = df \
    .withColumn('event_type',   F.lower(F.trim(F.col('event_type')))) \
    .withColumn('order_status', F.lower(F.trim(F.col('order_status')))) \
    .withColumn('channel',      F.lower(F.trim(F.col('channel')))) \
    .withColumn('currency',     F.upper(F.trim(F.col('currency'))))

# Step 3: Bad rows — message_id and order_id are required
# customer_id and amount_pence are optional
bad = df.filter(
    F.col('message_id').isNull() |
    F.col('order_id').isNull() |
    F.col('event_ts').isNull() |
    ~F.col('event_type').isin(VALID_EVENT_TYPES) |
    ~F.col('order_status').isin(VALID_STATUSES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('order_events'))

# Step 4: Good rows
good = df.filter(
    F.col('message_id').isNotNull() &
    F.col('order_id').isNotNull() &
    F.col('event_ts').isNotNull() &
    F.col('event_type').isin(VALID_EVENT_TYPES) &
    F.col('order_status').isin(VALID_STATUSES)
)

w = Window.partitionBy('message_id').orderBy(F.col('event_ts').desc())
good = good.withColumn('_rn', F.row_number().over(w)) \
           .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad.count()
good_count = good.count()
print(f'SQS events: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Check event type distribution
print('\nEvent type distribution (before filter):')
df.groupBy('event_type').count().orderBy('count', ascending=False).show()

# Step 5: Write
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.message_id = s.message_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('MERGE complete')
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
    print('Initial load complete')

# Step 6: Quarantine
if bad_count > 0:
    quarantine = bad.select(
        F.lit('src_05_sqs').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    )
    quarantine.write.format('delta').mode('append') \
        .option('mergeSchema', 'true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} rows quarantined')
else:
    print('No quarantine rows')


In [0]:
# ── VERIFY ────────────────────────────────────────────────────
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_05_sqs.order_events: {count} rows')
spark.sql(f"""
    SELECT event_type, order_status, COUNT(*) as cnt
    FROM {TARGET_TABLE}
    GROUP BY event_type, order_status
    ORDER BY cnt DESC
""").show()
